# Tag 12 – Pipeline-Orchestrierung mit Prefect (Live-Demo)

**Hinweis:** Diese Notebook-Version deckt die Python-API-Teile des Vortrags ab (Abschnitte 4–7). Das Prefect-Dashboard selbst (`uv run prefect server start`, `http://localhost:4200`) läuft weiterhin als eigener Prozess im Terminal, parallel zu diesem Notebook.

Ziel: Flow/Task-Grundkonzepte am eigenen Code sehen, die bestehenden Tag-9-Bausteine (Validierung, Training) zu einer Pipeline zusammensetzen, und Retry-Logik live erleben.

## Schritt 1: Flow und Task – die Grundbausteine

- **Task**: ein einzelner Schritt
- **Flow**: die gesamte Pipeline, die Tasks in der richtigen Reihenfolge aufruft

Kein neues Konzept, keine neue Sprache – normale Python-Funktionen mit einem Dekorator davor.

In [1]:
from prefect import flow, task

@task
def validate():
    print("Validiere Daten...")

@task
def train():
    print("Trainiere Modell...")

@flow
def pipeline():
    validate()
    train()

pipeline()

08:21:18.942 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8057
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

08:21:22.487 | INFO    | Flow run 'nondescript-spider' - Beginning flow run 'nondescript-spider' for flow 'pipeline'

Validiere Daten...


08:21:22.509 | INFO    | Task run 'validate-9a5' - Finished in state Completed()

Trainiere Modell...


08:21:22.515 | INFO    | Task run 'train-78f' - Finished in state Completed()

08:21:23.507 | INFO    | Flow run 'nondescript-spider' - Finished in state Completed()

    **Was ist gerade passiert?** Prefect hat im Hintergrund eine kleine, lokale API gestartet (falls nicht schon ein `prefect server` läuft), den Flow-Run ausgeführt und dabei Status, Timing und Logs jedes einzelnen Tasks mitgeschrieben.

Läuft parallel ein `uv run prefect server start` im Terminal, seht ihr genau diesen Run live im Dashboard unter `http://localhost:4200`.

## Schritt 2: Die echte Pipeline – Validierung → Training → Tracking

Jetzt keine Platzhalter-Funktionen mehr, sondern die tatsächlichen Bausteine aus Tag 9: die Pandera-Validierung und das MLflow-getrackte Training. Kein neuer Code – nur eine neue Hülle drumherum.

In [2]:
from pathlib import Path

import pandas as pd
import pandera.pandas as pa
from pandera import Column, Check
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

mlflow.set_experiment("iris-klassifikator")

SCHEMA = pa.DataFrameSchema({
    "sepal_length": Column(float, Check.in_range(0, 10)),
    "sepal_width": Column(float, Check.in_range(0, 10)),
    "petal_length": Column(float, Check.in_range(0, 10)),
    "petal_width": Column(float, Check.in_range(0, 10)),
    "target": Column(int, Check.isin([0, 1, 2])),
})

CURRENT_DIR: Path = Path.cwd()
DATA_FILE: Path = CURRENT_DIR / "data" / "iris.csv"


def load_dataset(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(
            f"'{file_path}' nicht gefunden - erst Tag 9 durchlaufen "
            "(dvc add bzw. export_data.py)."
        )
    return pd.read_csv(filepath_or_buffer=file_path)


@task
def validate_data(file_path: Path) -> pd.DataFrame:
    df = load_dataset(file_path=file_path)
    SCHEMA.validate(df)
    print(f"Validierung erfolgreich: {len(df)} Zeilen.")
    return df


@task
def train_model(df: pd.DataFrame, n_estimators: int = 100) -> float:
    X = df.drop(columns=["target"])
    y = df["target"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    with mlflow.start_run(run_name=f"prefect-flow-{n_estimators}"):
        mlflow.log_param("n_estimators", n_estimators)
        model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))
        mlflow.log_metric("accuracy", accuracy)
        print(f"Training abgeschlossen: accuracy={accuracy:.3f}")
    return accuracy


@flow
def training_pipeline(file_path: Path, n_estimators: int = 100) -> float:
    df = validate_data(file_path=file_path)
    return train_model(df=df, n_estimators=n_estimators)


training_pipeline(file_path=DATA_FILE, n_estimators=150)

08:21:29.799 | INFO    | Flow run 'shaggy-nuthatch' - Beginning flow run 'shaggy-nuthatch' for flow 'training-pipeline'

Validierung erfolgreich: 150 Zeilen.


08:21:29.828 | INFO    | Task run 'validate_data-eb8' - Finished in state Completed()

Training abgeschlossen: accuracy=1.000
🏃 View run prefect-flow-150 at: http://localhost:5000/#/experiments/1/runs/dcf3cb425b7c4fa4bd055c928fd10110
🧪 View experiment at: http://localhost:5000/#/experiments/1


08:21:30.264 | INFO    | Task run 'train_model-f94' - Finished in state Completed()

08:21:30.802 | INFO    | Flow run 'shaggy-nuthatch' - Finished in state Completed()

1.0

## Schritt 3: Retry-Logik bei instabilen Schritten

Manche Schritte scheitern gelegentlich aus Gründen, die nichts mit eurem Code zu tun haben – ein kurzer Netzwerkaussetzer, ein überlasteter Dienst. Statt die ganze Pipeline abbrechen zu lassen, kann Prefect automatisch erneut versuchen.

Zur Veranschaulichung: ein Task, der zufällig manchmal fehlschlägt.

In [5]:
import random

@task(retries=3, retry_delay_seconds=5)
def instabiler_schritt() -> str:
    if random.random() < 0.6:
        raise RuntimeError("Simulierter, kurzzeitiger Fehler")
    print("Instabiler Schritt erfolgreich (evtl. erst nach Retry).")
    return "ok"


@flow
def flow_mit_retry() -> str:
    return instabiler_schritt()


flow_mit_retry()

08:23:46.740 | INFO    | Flow run 'talkative-loris' - Beginning flow run 'talkative-loris' for flow 'flow-mit-retry'

08:23:46.749 | INFO    | Task run 'instabiler_schritt-40f' - Task run failed with exception: RuntimeError('Simulierter, kurzzeitiger Fehler') - Retry 1/3 will start 5 second(s) from now

08:23:51.751 | INFO    | Task run 'instabiler_schritt-40f' - Task run failed with exception: RuntimeError('Simulierter, kurzzeitiger Fehler') - Retry 2/3 will start 5 second(s) from now

08:23:56.753 | INFO    | Task run 'instabiler_schritt-40f' - Task run failed with exception: RuntimeError('Simulierter, kurzzeitiger Fehler') - Retry 3/3 will start 5 second(s) from now

08:24:01.755 | ERROR   | Task run 'instabiler_schritt-40f' - Task run failed with exception: RuntimeError('Simulierter, kurzzeitiger Fehler') - Retries are exhausted
Traceback (most recent call last):
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1021, in run_context
    yield self
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1684, in run_task_sync
    engine.call_task_fn(txn)
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1038, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_80111/347932045.py", line 6, in instabiler_schritt
    raise RuntimeError("Simulierter, kurzzeitiger Fehler")
RuntimeError: Simulierter, kurzzeitiger Fehler

08:24:01.760 | ERROR   | Task run 'instabiler_schritt-40f' - Finished in state Failed('Task run encountered an exception RuntimeError: Simulierter, kurzzeitiger Fehler')

08:24:02.745 | ERROR   | Flow run 'talkative-loris' - Encountered exception during execution: RuntimeError('Simulierter, kurzzeitiger Fehler')
Traceback (most recent call last):
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/flow_engine.py", line 1330, in run_context
    yield self
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/flow_engine.py", line 2111, in run_flow_sync
    engine.call_flow_fn()
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/flow_engine.py", line 1350, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_80111/347932045.py", line 13, in flow_mit_retry
    return instabiler_schritt()
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/tasks.py", line 1227, in __call__
    return run_task(
           ^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1917, in run_task
    return run_task_sync(**kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1687, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 617, in result
    raise self._raised
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1021, in run_context
    yield self
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1684, in run_task_sync
    engine.call_task_fn(txn)
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/task_engine.py", line 1038, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m/mlops-projekt/.venv/lib/python3.12/site-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_80111/347932045.py", line 6, in instabiler_schritt
    raise RuntimeError("Simulierter, kurzzeitiger Fehler")
RuntimeError: Simulierter, kurzzeitiger Fehler

08:24:02.816 | INFO    | Flow run 'talkative-loris' - Finished in state Failed('Flow run encountered an exception: RuntimeError: Simulierter, kurzzeitiger Fehler')

RuntimeError: Simulierter, kurzzeitiger Fehler

**Was passiert im Hintergrund:** Schlägt der Task fehl, wartet Prefect 5 Sekunden und versucht es erneut – bis zu dreimal. Erst wenn auch der letzte Versuch scheitert, gilt der Task als endgültig fehlgeschlagen. Im Dashboard seht ihr jeden einzelnen Versuch als eigenen Log-Eintrag.

## Einordnung: Batch-Inference

Zur Abgrenzung – was wir hier bauen, ist **nicht** dasselbe wie die Inference-API aus Tag 4/14. Dort: eine einzelne Anfrage in Echtzeit. Eine Batch-Inference-Pipeline würde stattdessen das Modell einmal auf einen kompletten, neuen Datensatz gleichzeitig anwenden – z. B. wöchentlich alle neu hinzugekommenen Datensätze durchscoren. Strukturell wäre das ein weiterer `@task`, der `model.predict(neue_daten)` statt `model.fit(...)` aufruft.

## Zusammenfassung

- **Task** = ein Schritt, **Flow** = die ganze Pipeline
- Bestehender Code wird durch Dekoratoren zu Pipeline-Bausteinen – keine Neuimplementierung nötig
- `retries`/`retry_delay_seconds` fangen kurzzeitige Fehler automatisch ab
- Scheduling (zeitgesteuerte Flows) und Deployments sind Vertiefung im Selbststudium

**Weiter geht's:** In der Übung baut ihr genau diese Pipeline für euer eigenes Projekt und ergänzt einen Batch-Inference-Flow.